# 案例3：酷狗音乐华语新歌榜爬取

爬取酷狗音乐华语新歌榜中的歌曲信息，提取歌曲名称、歌手和播放地址，并保存为 CSV 文件。

## 一、导入库

使用 Requests 获取网页，使用 lxml 和 XPath 解析页面，使用 Pandas 保存结构化数据。

In [ ]:
%pip install requests lxml pandas

In [1]:
import requests
from lxml import etree
import pandas as pd

## 二、获取网页

`get_html()` 发送 GET 请求，设置网页编码，并检查请求状态。

In [2]:
def get_html(url, time=30):
    try:
        r = requests.get(url, timeout=time)
        r.encoding = r.apparent_encoding
        r.raise_for_status()
        return r.text
    except Exception as error:
        print(error)

## 三、解析歌曲信息

根据教材中的页面结构，歌曲列表位于 `id="rankWrap"` 的 div 中。歌曲信息保存在 li 标签的 `title` 属性中，播放链接保存在 a 标签的 `href` 属性中。

In [3]:
def parser(html):
    doc = etree.HTML(html)

    title_list = doc.xpath("//div[@id='rankWrap']//li/@title")
    href_list = doc.xpath("//div[@id='rankWrap']//li/a/@href")

    out_dict = {
        "歌曲名称": [],
        "歌手": [],
        "播放地址": []
    }

    for title, href in zip(title_list, href_list):
        singer, song = title.split(" - ", 1)
        out_dict["歌曲名称"].append(song.strip())
        out_dict["歌手"].append(singer.strip())
        out_dict["播放地址"].append(href)

    return out_dict

## 四、保存为 CSV 文件

将解析结果转换为 DataFrame，再保存为 CSV 文件。

In [4]:
def save_dict2csv(item, path):
    df = pd.DataFrame(item)
    df.to_csv(path, index=False, encoding="utf-8-sig")

## 五、运行程序

目标网址为教材提供的酷狗音乐华语新歌榜页面。

In [5]:
url = "http://www.bspider.top/kugou/"
html = get_html(url)
out_dict = parser(html)

print("爬取歌曲数量：", len(out_dict["歌曲名称"]))
print("歌曲信息：")
for i in range(min(5, len(out_dict["歌曲名称"]))):
    print(out_dict["歌曲名称"][i], out_dict["歌手"][i], out_dict["播放地址"][i])

save_dict2csv(out_dict, "music20250930.csv")

爬取歌曲数量： 22
歌曲信息：
日出时让街灯安睡 李幸倪、张学友 https://www.kugou.com/song/1wyaktc2.html
无人与我 (粤语版) 亮声open https://www.kugou.com/song/1wv0zg5e.html
朱雀 时代少年团 https://www.kugou.com/song/1wr89r02.html
吴哥窟 (莫叫语版) 莫叫姐姐 https://www.kugou.com/song/1wr86kf3.html
你们 岳云鹏 https://www.kugou.com/song/1ws0hmf9.html
